# Annotated variants with CAID and VEP output

In [ ]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [ ]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)


parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        clinvar_processed_rawdata=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_rawdata_6-24-24.txt", sep="\t")
        print("input file")
        print(len(clinvar_processed_rawdata))
        clinvar_rawdata_filter_PLPconflict_HGVSc_reset_index=clinvar_processed_rawdata.set_index('HGVSc_new')
                
        !grep -v "##" Python_coderun_6-13-24_clinvar_extract_and_process_again/commandlineVEPv112_output_August2024.txt > Python_coderun_6-13-24_clinvar_extract_and_process_again/commandlineVEPv112_output_August2024_noinfo.txt
        clinvar_VEPoutput_MANE = pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/commandlineVEPv112_output_August2024_noinfo.txt", sep="\t")
        print(len(clinvar_VEPoutput_MANE))
        display(clinvar_VEPoutput_MANE)
        
        clinvar_VEPoutput_MANE_filtertranscript=clinvar_VEPoutput_MANE[clinvar_VEPoutput_MANE["Feature_type"].str.contains("Transcript")==True]
        
        print(len(clinvar_VEPoutput_MANE_filtertranscript))
        display(clinvar_VEPoutput_MANE_filtertranscript)
        
        
        #drop duplicate variants for this step
        clinvar_VEPoutput_MANE_unique=clinvar_VEPoutput_MANE_filtertranscript.drop_duplicates(subset=['#Uploaded_variation'])
        print(len(clinvar_VEPoutput_MANE_unique))
        
        #collapse categories - to 1st most severe annotation, for sg,fs collapse to fs and for splice x 3 replace with splice region:
        consequence_list=[]
        clinvar_VEPoutput_MANE_unique.insert(1,"collapsed_Consequence","NaN")
        clinvar_VEPoutput_MANE_unique.insert(1,"AlleleLength","NaN")
        for index, row in clinvar_VEPoutput_MANE_unique.iterrows():
            rowindex=clinvar_VEPoutput_MANE_unique.index.get_loc(index)
            colindex_consequence=clinvar_VEPoutput_MANE_unique.columns.get_loc('collapsed_Consequence')
            colindex_allelelength=clinvar_VEPoutput_MANE_unique.columns.get_loc('AlleleLength')
            consequence_list=str(clinvar_VEPoutput_MANE_unique.loc[index]["Consequence"]).split(',')
            if ("stop_gained" in consequence_list) & ("frameshift_variant" in consequence_list):
                clinvar_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=consequence_list[1]
            else:
                if (consequence_list[0]=="splice_donor_region_variant")|(consequence_list[0]=="splice_donor_5th_base_variant")|(consequence_list[0]=="splice_polypyrimidine_tract_variant"):
                    consequence_list[0]="splice_region_variant"
                clinvar_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=consequence_list[0]
            #replace collapsed category with ">=50bp_indel" if lareg insertion/deletion as name suggests greater than or equal to 50 bp (measured as difference between length of reference allele and length of alternate allele)
            clinvar_VEPoutput_MANE_unique.iloc[rowindex,colindex_allelelength]=len(clinvar_VEPoutput_MANE_unique.loc[index]["Allele"])-len(clinvar_VEPoutput_MANE_unique.loc[index]["REF_ALLELE"])
            if (abs(clinvar_VEPoutput_MANE_unique.loc[index]['AlleleLength']) >= 50):
                clinvar_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=">=50bp_indel"
        
        #set HGVSc as index column
        clinvar_VEPoutput_MANE_unique_resetindex=clinvar_VEPoutput_MANE_unique.set_index('#Uploaded_variation')
        
        
        ###CAR API for CA ID:
        url_CARAPI="https://reg.clinicalgenome.org/alleles?file=hgvs&fields=none+@id"
        res_CARAPI=requests.post(url_CARAPI,data=open('Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_rawdata_filter_PLPconflict_HGVSctoannotate_6-24-24.txt').read())
        CARAPI_output=pd.read_json(res_CARAPI.text)
        display(CARAPI_output)


        #join CARAPI output back to dataframe first by joining to input hgvsg since expected to be in same order, and then using hgvsg column to join back to processed raw data file:
        #read HGVSg file to df
        clinvar_VEP_unique_hgvs = pd.read_csv('Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_rawdata_filter_PLPconflict_HGVSctoannotate_6-24-24.txt', sep="\t", header=None)
        #join CAR API output to HGVSg input file by numerical index
        clinvar_VEP_unique_HGVSg_CARAPI=clinvar_VEP_unique_hgvs.join(CARAPI_output)
        display(clinvar_VEP_unique_HGVSg_CARAPI)
        #drop 1st row with error since 'HGVSg' header in first row
        #clinvar_VEP_unique_HGVSg_CARAPI=clinvar_VEP_unique_HGVSg_CARAPI.drop(0)
        #rename HGVSg column from input file joined to output file
        #cbio_VEP_unique_HGVSg_CARAPI_rename_unique=cbio_VEP_unique_HGVSg_CARAPI.rename(columns={0:"HGVS_input"})
        #reset index to HGVSg column to join with that processed rawdata file
        #cbio_VEP_unique_HGVSg_CARAPIunique_resetindex = cbio_VEP_unique_HGVSg_CARAPI_rename_unique.set_index('@id')

                    
        #drop rows which had 'error' output from CAR API (QC if there is an error and, df output at this step to check what and why)
        clinvar_VEP_unique_HGVSg_CARAPI_errors=clinvar_VEP_unique_HGVSg_CARAPI[clinvar_VEP_unique_HGVSg_CARAPI['@id'].str.contains("error")==True]
        #cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_no_errors = cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.drop(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.index)
        #cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.to_csv("Python_coderun_8-24-23/Python_cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.txt", sep="\t")
        print(len(clinvar_VEP_unique_HGVSg_CARAPI_errors))  
        #clinvar_VEP_unique_HGVSg_CARAPI.to_csv("clinvar_unique_CARAPI_CAID_7-12-24.txt", sep="\t")
        
        clinvar_VEP_unique_HGVSg_CARAPI_setindex=clinvar_VEP_unique_HGVSg_CARAPI.set_index(0)
        
        
        #clinvar_VEP_unique_HGVSg_CARAPI=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/clinvar_unique_CARAPI_CAID_7-12-24.txt", sep="\t")
        
        #clinvar_VEP_unique_HGVSg_CARAPI_setindex=clinvar_VEP_unique_HGVSg_CARAPI.set_index("0")
        
        #join by HGVSc index columns
        
        clinvar_VEP_CARAPI=clinvar_VEPoutput_MANE_unique_resetindex.join(clinvar_VEP_unique_HGVSg_CARAPI_setindex)
        print("VEP CA ID annotated unique")
        print(len(clinvar_VEP_CARAPI))
        display(clinvar_VEP_CARAPI)
        
        clinvar_rawdata_VEP_MANE=clinvar_rawdata_filter_PLPconflict_HGVSc_reset_index.join(clinvar_VEP_CARAPI, rsuffix="clinvar_VEP_CAID_df")
        #reset index back to numbers so that no duplicate indices (due to repeat variants)
        clinvar_rawdata_VEP_MANE_resetindex=clinvar_rawdata_VEP_MANE.reset_index()
        print(f"clinvar_VEP_CAID len={len(clinvar_rawdata_VEP_MANE_resetindex)}")
        display(clinvar_rawdata_VEP_MANE_resetindex)
        
        #remove lines where no VEP output, and write to file to QC why they had no VEP output
        clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan=clinvar_rawdata_VEP_MANE_resetindex[clinvar_rawdata_VEP_MANE_resetindex["Consequence"].isnull()]
        #clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan.to_csv("Python_coderun_8-24-23/Python_clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan_forQC.txt", sep="\t")  
        clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_dropna = clinvar_rawdata_VEP_MANE_resetindex.drop(clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan.index)
        #print(len(clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE))
        print(len(clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan))
        print(len(clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_dropna))
        
        print(f"clinvar_VEP len={len(clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_dropna)}")
        clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_dropna.to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_VEP_MANE_8-15-24.txt", sep="\t")
        clinvar_rawdata_filter_PLPconflict_HGVSc_VEP_MANE_nan.to_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_VEP_MANE_nan_forQC_8-15-24.txt", sep="\t")
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



In [ ]:
## 8-16-24  annotating cbio file with VEP from commandline v112 run

start_time=time.asctime(time.localtime())
print(start_time)
#setting path to inpu12t and output files
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

# Loop through each folder name
for index, row in TranscriptID.iterrows():
    
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        cbio_processed_rawdata=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_rawdata.txt", sep="\t")
        print(len(cbio_processed_rawdata))
        
        os.chdir("Python_coderun_7-11-24_cbio_process_again")
        #read cBio VEPoutput MANE file from VEP run and join with processed file:
        !grep -v "##" commandlineVEPv112_output_August2024.txt > commandlineVEPv112_output_August2024_noinfo.txt
        
        
        cbio_VEPoutput_MANE = pd.read_csv("commandlineVEPv112_output_August2024_noinfo.txt", sep="\t")
        print("VEP file")
        print(len(cbio_VEPoutput_MANE))
        
        cbio_VEPoutput_MANE_filtertranscript=cbio_VEPoutput_MANE[cbio_VEPoutput_MANE["Feature_type"].str.contains("Transcript")==True]
        
        print(len(cbio_VEPoutput_MANE_filtertranscript))
        display(cbio_VEPoutput_MANE_filtertranscript)
        
        
        #reset index to HGVSc column to join by this column to VEP output
        cbio_rawdata_s_u_p_olo_hgvsc=cbio_processed_rawdata.reset_index().set_index('HGVSc')
        #drop duplicates for this step
        cbio_VEPoutput_MANE_unique=cbio_VEPoutput_MANE_filtertranscript.drop_duplicates(subset=['#Uploaded_variation'])
        print(len(cbio_VEPoutput_MANE_unique))
        
        #collapse categories - to 1st most severe annotation, for stop-gain,frame-shift collapse to frame-shift and for splice x 3 replace with splice region:
        consequence_list=[]
        cbio_VEPoutput_MANE_unique.insert(1,"collapsed_Consequence","NaN")
        cbio_VEPoutput_MANE_unique.insert(1,"AlleleLength","NaN")
        for index, row in cbio_VEPoutput_MANE_unique.iterrows():
            rowindex=cbio_VEPoutput_MANE_unique.index.get_loc(index)
            colindex_consequence=cbio_VEPoutput_MANE_unique.columns.get_loc('collapsed_Consequence')
            colindex_allelelength=cbio_VEPoutput_MANE_unique.columns.get_loc('AlleleLength')
            consequence_list=str(cbio_VEPoutput_MANE_unique.loc[index]["Consequence"]).split(',')
            if ("stop_gained" in consequence_list) & ("frameshift_variant" in consequence_list):
                cbio_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=consequence_list[1]
            else:
                if (consequence_list[0]=="splice_donor_region_variant")|(consequence_list[0]=="splice_donor_5th_base_variant")|(consequence_list[0]=="splice_polypyrimidine_tract_variant"):
                    consequence_list[0]="splice_region_variant"
                cbio_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=consequence_list[0]
            #replace collapsed category with ">=50bp_indel" if lareg insertion/deletion as name suggests greater than or equal to 50 bp (measured as difference between length of reference allele and length of alternate allele)
            cbio_VEPoutput_MANE_unique.iloc[rowindex,colindex_allelelength]=len(cbio_VEPoutput_MANE_unique.loc[index]["Allele"])-len(cbio_VEPoutput_MANE_unique.loc[index]["REF_ALLELE"])
            if (abs(cbio_VEPoutput_MANE_unique.loc[index]['AlleleLength']) >= 50):
                cbio_VEPoutput_MANE_unique.iloc[rowindex,colindex_consequence]=">=50bp_indel"
         
        cbio_VEPoutput_MANE_unique_resetindex=cbio_VEPoutput_MANE_unique.set_index('#Uploaded_variation')

        ###CAR API for CA ID:
        
        url_CARAPI="https://reg.clinicalgenome.org/alleles?file=hgvs&fields=none+@id"
        res_CARAPI=requests.post(url_CARAPI,data=open('Python_cbio_processed_rawdata_unique_HGVSctoannotate.txt').read())
        CARAPI_output=pd.read_json(res_CARAPI.text)


        #join CARAPI output back to cbio dataframe first by joining to input hgvsg since expected to be in same order, and then using hgvsg column to join back to processed raw data file:
        #read HGVSg file to df
        cbio_VEP_unique_hgvs = pd.read_csv('Python_cbio_processed_rawdata_unique_HGVSctoannotate.txt', sep="\t", header=None)
        #join CAR API output to HGVSg input file by numerical index
        cbio_VEP_unique_HGVSg_CARAPI=cbio_VEP_unique_hgvs.join(CARAPI_output)
        #drop 1st row with error since 'HGVSg' header in first row- not the case anymore so need to remove this code. and re run CAID code. ######### 8-16-24 #######
        #cbio_VEP_unique_HGVSg_CARAPI=cbio_VEP_unique_HGVSg_CARAPI.drop(0)
        #rename HGVSg column from input file joined to output file
        #cbio_VEP_unique_HGVSg_CARAPI_rename_unique=cbio_VEP_unique_HGVSg_CARAPI.rename(columns={0:"HGVS_input"})
        #reset index to HGVSg column to join with that processed rawdata file
        #cbio_VEP_unique_HGVSg_CARAPIunique_resetindex = cbio_VEP_unique_HGVSg_CARAPI_rename_unique.set_index('@id')
        display(cbio_VEP_unique_HGVSg_CARAPI)
                    
        #drop rows which had 'error' output from CAR API (QC if there is an error and, df output at this step to check what and why)
        cbio_VEP_unique_HGVSg_CARAPI_errors=cbio_VEP_unique_HGVSg_CARAPI[cbio_VEP_unique_HGVSg_CARAPI['@id'].str.contains("error")==True]
        #cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_no_errors = cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.drop(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.index)
        #cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.to_csv("Python_coderun_8-24-23/Python_cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.txt", sep="\t")
        print(len(cbio_VEP_unique_HGVSg_CARAPI_errors))  
        cbio_VEP_unique_HGVSg_CARAPI.to_csv("cbio_unique_CARAPI_CAID_8-16-24.txt", sep="\t")
        
        cbio_VEP_unique_HGVSg_CARAPI_setindex=cbio_VEP_unique_HGVSg_CARAPI.set_index(0)
        
        #join by HGVSc index columns
        
        cbio_VEP_CARAPI=cbio_VEPoutput_MANE_unique_resetindex.join(cbio_VEP_unique_HGVSg_CARAPI_setindex)
        print("VEP CA ID annotated unique")
        print(len(cbio_VEP_CARAPI))
        display(cbio_VEP_CARAPI)
        
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE=cbio_rawdata_s_u_p_olo_hgvsc.join(cbio_VEP_CARAPI, rsuffix="cbio_VEP_CAID_df")
        #reset index back to numbers so that no duplicate indices (due to repeat variants)
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex=cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE.reset_index()
        print(f"cbio_VEP_CAID len={len(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex)}")
        display(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex)
        
        #remove lines where no VEP output, and write to file to QC why they had no VEP output
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_nan=cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex[cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex["Consequence"].isnull()]
        
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_dropna = cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex.drop(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_nan.index)
        
        print(len(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_nan))
        print(len(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_dropna))
        
        print(f"cbio_VEP_CAID_dropna len={len(cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_dropna)}")
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_dropna.to_csv("Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")
        cbio_rawdata_s_u_p_olo_hgvsc_VEP_MANE_resetindex_nan.to_csv("Python_cbio_processed_VEP_MANE_nan_forQC_8-16-24.txt", sep="\t")
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)     

# Getting counts of coding variants in ClinVar for Figure 1

In [58]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)


parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

clinvarcount=clinvarcodingcount=clinvaroccurrencetotal=0
oldclinvaroccurrencetotal=0
for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        clinvar=pd.read_csv("Python_coderun_6-13-24_clinvar_extract_and_process_again/Python_clinvar_processed_VEP_MANE_8-15-24.txt", sep="\t")

        
        ReviewStatustoremove=["no classification provided","no classification for the individual variant"]
        clinvar_reviewstatustoremove=clinvar[clinvar["ReviewStatus"].isin(ReviewStatustoremove)]
        print("QC_reviewstatus")
        print(len(clinvar_reviewstatustoremove))
        clinvarcount=clinvarcount+len(clinvar)
        oldclinvaroccurrencetotal=oldclinvaroccurrencetotal+clinvar["Occurrence"].sum()
        
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"]
        #cbio_CAID=cbio_CAID[~cbio_CAID["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        clinvar_dropnoncoding=clinvar[~clinvar["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        print("QC_codingonly")
        print(len(clinvar)-len(clinvar_dropnoncoding))
        clinvarcodingcount=clinvarcodingcount+len(clinvar_dropnoncoding)
        clinvaroccurrencetotal=clinvaroccurrencetotal+clinvar_dropnoncoding["Occurrence"].sum()
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



Tue Apr 29 16:49:21 2025
QC_reviewstatus
0
QC_codingonly
2
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
QC_reviewstatus
0
QC_codingonly
6
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL
QC_reviewstatus
0
QC_codingonly
30
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC2
QC_reviewstatus
0
QC_codingonly
8
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC1
QC_reviewstatus
0
QC_codingonly
10
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TP53
QC_reviews

In [61]:
clinvarcodingcount

32941

In [62]:
clinvaroccurrencetotal

77247